# Experiment 41 v3 — Clean Audit of Downstream Significance Metadata

The v2 run revealed that significance information is in fact widely available, but the final
coverage report was wrong because `condition_level_selected.csv` already contained columns named
`CI_Low`, `CI_High`, etc. Merging a second significance table created `CI_Low_x` / `CI_Low_y`,
while the reporting cell looked for the unsuffixed `CI_Low`.

A second issue is that a recursive scan can rediscover files written by the recovery notebook itself,
creating circular provenance.

This v3 notebook fixes both issues.

## Rules

1. The Experiment 37 selected 96-condition table is the authoritative condition list.
2. Significance already stored in that table is retained first.
3. Historical bootstrap/CI files are scanned only outside the recovery/meta-analysis output folders.
4. Missing values are filled from the best matching historical source; existing selected values are
   never overwritten.
5. No significance label is inferred merely from the sign of MSE gain.
6. Every recovered interval must have a direction consistent with the frozen MSE gain.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 200)

DATASETS = [
    "Solar",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "ETTh1",
]

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
    "SegMoE",
]

HORIZONS = [96, 192, 336, 720]

ROOT_CANDIDATES = [
    Path("/data/dataset/strong_forecaster"),
    Path("/data/strong_forecaster"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)

if ROOT is None:
    raise FileNotFoundError("strong_forecaster root not found")

OUT_DIR = ROOT / "significance_metadata_recovery_v3_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


ROOT: /data/dataset/strong_forecaster
OUT_DIR: /data/dataset/strong_forecaster/significance_metadata_recovery_v3_clean


## 1. Load the frozen Experiment 37 condition table

In [2]:
preferred = (
    ROOT
    / "four_backbone_dataset_meta_analysis"
    / "condition_level_selected.csv"
)

if not preferred.is_file():
    hits = sorted(ROOT.rglob("condition_level_selected.csv"))
    hits = [
        p for p in hits
        if "significance_metadata_recovery" not in str(p)
    ]

    if not hits:
        raise FileNotFoundError("condition_level_selected.csv not found")

    preferred = hits[0]

conditions = pd.read_csv(preferred)

required = {
    "Dataset",
    "Backbone",
    "Horizon",
    "MSEGain_pct",
}

missing = required - set(conditions.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

conditions = conditions[
    conditions["Dataset"].isin(DATASETS)
    & conditions["Backbone"].isin(BACKBONES)
    & conditions["Horizon"].astype(int).isin(HORIZONS)
].copy()

conditions["Horizon"] = conditions["Horizon"].astype(int)

KEY = ["Dataset", "Backbone", "Horizon"]

if len(conditions) != 96:
    raise RuntimeError(f"Expected 96 conditions; got {len(conditions)}")

if conditions.duplicated(KEY).any():
    raise RuntimeError("Duplicate selected conditions found")

print("Condition table:", preferred)
print("Conditions:", len(conditions))
print("Columns:", list(conditions.columns))


Condition table: /data/dataset/strong_forecaster/four_backbone_dataset_meta_analysis/condition_level_selected.csv
Conditions: 96
Columns: ['Dataset', 'Backbone', 'Horizon', 'ProtocolFamily', 'ExperimentFamily', 'Direct_MSE', 'Final_MSE', 'MSEGain_pct', 'Direct_MAE', 'Final_MAE', 'MAEGain_pct', 'FinalMeanAlpha', 'OracleHeadroom_pct', 'ValidationGain_pct', 'CI_Low', 'CI_High', 'SignificantPositive', 'SignificantNegative', 'SourcePath', 'SourceScore', 'RecoveredFamilyBonus', 'FinalSourceScore']


## 2. Audit significance already present in the selected table

This step is important because Experiment 37 already propagated significance metadata for many
conditions. We preserve those exact values before searching historical artifacts.


In [3]:
SIG_COLS = [
    "CI_Low",
    "CI_High",
    "SignificantPositive",
    "SignificantNegative",
]

for c in SIG_COLS:
    if c not in conditions.columns:
        conditions[c] = np.nan

base = conditions.copy()

base["SelectedHasCI"] = (
    pd.to_numeric(base["CI_Low"], errors="coerce").notna()
    & pd.to_numeric(base["CI_High"], errors="coerce").notna()
)

print(
    "Significance already present in selected table:",
    int(base["SelectedHasCI"].sum()),
    "/96",
)

display(
    base.groupby("Dataset")
    .agg(
        Conditions=("Horizon", "size"),
        SelectedWithCI=("SelectedHasCI", "sum"),
    )
    .reset_index()
)


Significance already present in selected table: 68 /96


,Dataset,Conditions,SelectedWithCI
0,ETTh1,16,16
1,Electricity,16,12
2,Exchange,16,4
3,Solar,16,4
4,Traffic,16,16
5,Weather,16,16


## 3. Canonicalization and clean source filtering

In [4]:
def canon_dataset(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")
    return {
        "solar": "Solar",
        "solarenergy": "Solar",
        "weather": "Weather",
        "electricity": "Electricity",
        "ecl": "Electricity",
        "traffic": "Traffic",
        "exchange": "Exchange",
        "exchangerate": "Exchange",
        "etth1": "ETTh1",
    }.get(s, str(x).strip())


def canon_backbone(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")
    return {
        "patchtst": "PatchTST",
        "itransformer": "iTransformer",
        "timemixer": "TimeMixer",
        "segmoe": "SegMoE",
        "segmoeforecast": "SegMoE",
    }.get(s, str(x).strip())


def find_col(df, names):
    lower = {str(c).lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None


def as_bool(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return bool(v)
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return np.nan


def is_excluded_path(p):
    s = str(p)

    # Never read outputs from this recovery workflow back into itself.
    banned = [
        "significance_metadata_recovery/",
        "significance_metadata_recovery_v3_clean/",
    ]

    return any(token in s for token in banned)


def source_priority(p):
    name = p.name.lower()
    s = str(p).lower()

    if name == "bootstrap.csv":
        return 0

    if "with_bootstrap" in name or "bootstrap" in name:
        return 1

    # Derived evidence tables are useful only after true bootstrap files.
    if (
        "cross_backbone_integrated_evidence" in s
        or "three_backbone_integrated_evidence" in s
    ):
        return 3

    return 2


## 4. Scan historical significance files, excluding recovery outputs


In [5]:
candidate_files = []

for p in ROOT.rglob("*.csv"):
    if is_excluded_path(p):
        continue

    # The Experiment 37 selected table is handled separately above.
    if p.resolve() == preferred.resolve():
        continue

    try:
        size_mb = p.stat().st_size / (1024**2)
    except Exception:
        continue

    if size_mb > 500:
        continue

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(df, ["Dataset", "Data"])
    c_backbone = find_col(df, ["Backbone", "Model"])
    c_horizon = find_col(df, ["Horizon", "PredLen", "pred_len", "H"])
    c_lo = find_col(df, ["CI_Low", "CILow", "Lower", "Lower95", "CI95_Low"])
    c_hi = find_col(df, ["CI_High", "CIHigh", "Upper", "Upper95", "CI95_High"])

    if None in {c_dataset, c_backbone, c_horizon, c_lo, c_hi}:
        continue

    candidate_files.append({
        "Path": str(p),
        "Rows": len(df),
        "Priority": source_priority(p),
        "SizeMB": size_mb,
    })

inventory = pd.DataFrame(
    candidate_files,
    columns=["Path", "Rows", "Priority", "SizeMB"],
)

if len(inventory):
    inventory = (
        inventory
        .drop_duplicates("Path")
        .sort_values(["Priority", "Path"])
        .reset_index(drop=True)
    )

display(inventory)

inventory.to_csv(
    OUT_DIR / "clean_candidate_significance_files.csv",
    index=False,
)

print("Clean candidate files:", len(inventory))


,Path,Rows,Priority,SizeMB
0,/data/dataset/strong_forecaster/electricity_fu...,12,0,0.001682
1,/data/dataset/strong_forecaster/electricity_fu...,12,0,0.001809
2,/data/dataset/strong_forecaster/electricity_fu...,12,0,0.001769
3,/data/dataset/strong_forecaster/electricity_fu...,3,0,0.000540
4,/data/dataset/strong_forecaster/electricity_fu...,4,0,0.000662
5,/data/dataset/strong_forecaster/electricity_fu...,1,0,0.000257
6,/data/dataset/strong_forecaster/electricity_fu...,4,0,0.000668
7,/data/dataset/strong_forecaster/electricity_fu...,8,0,0.001174
8,/data/dataset/strong_forecaster/electricity_fu...,12,0,0.001710
9,/data/dataset/strong_forecaster/electricity_fu...,4,0,0.000654


Clean candidate files: 54


## 5. Parse candidate rows

In [6]:
parsed_rows = []

for _, meta in inventory.iterrows():
    p = Path(meta["Path"])

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(df, ["Dataset", "Data"])
    c_backbone = find_col(df, ["Backbone", "Model"])
    c_horizon = find_col(df, ["Horizon", "PredLen", "pred_len", "H"])

    c_comp = find_col(df, ["Comparison", "Compare", "MethodComparison"])
    c_mean = find_col(df, [
        "MeanImprovement",
        "MeanDiff",
        "MSEImprovement",
        "DeltaMSE",
        "MeanDifference",
    ])

    c_lo = find_col(df, ["CI_Low", "CILow", "Lower", "Lower95", "CI95_Low"])
    c_hi = find_col(df, ["CI_High", "CIHigh", "Upper", "Upper95", "CI95_High"])

    c_pos = find_col(df, [
        "SignificantPositive",
        "SigPositive",
        "PositiveSignificant",
    ])

    c_neg = find_col(df, [
        "SignificantNegative",
        "SigNegative",
        "NegativeSignificant",
    ])

    c_block = find_col(df, ["BlockLen", "BlockLength", "BootstrapBlockLen"])
    c_rep = find_col(df, ["Replicates", "NBoot", "BootstrapReplicates"])

    for idx, r in df.iterrows():
        dataset = canon_dataset(r[c_dataset])
        backbone = canon_backbone(r[c_backbone])

        try:
            horizon = int(r[c_horizon])
        except Exception:
            continue

        if dataset not in DATASETS:
            continue
        if backbone not in BACKBONES:
            continue
        if horizon not in HORIZONS:
            continue

        lo = pd.to_numeric(
            pd.Series([r[c_lo]]),
            errors="coerce",
        ).iloc[0]

        hi = pd.to_numeric(
            pd.Series([r[c_hi]]),
            errors="coerce",
        ).iloc[0]

        if pd.isna(lo) or pd.isna(hi):
            continue

        mean = (
            pd.to_numeric(
                pd.Series([r[c_mean]]),
                errors="coerce",
            ).iloc[0]
            if c_mean is not None
            else np.nan
        )

        comparison = (
            str(r[c_comp]).strip()
            if c_comp is not None and not pd.isna(r[c_comp])
            else ""
        )

        parsed_rows.append({
            "Dataset": dataset,
            "Backbone": backbone,
            "Horizon": horizon,
            "Comparison": comparison,
            "MeanImprovement": mean,
            "CI_Low_recovered": lo,
            "CI_High_recovered": hi,
            "SigPositive_recovered": (
                as_bool(r[c_pos])
                if c_pos is not None
                else bool(lo > 0)
            ),
            "SigNegative_recovered": (
                as_bool(r[c_neg])
                if c_neg is not None
                else bool(hi < 0)
            ),
            "BlockLen_recovered": (
                r[c_block] if c_block is not None else np.nan
            ),
            "Replicates_recovered": (
                r[c_rep] if c_rep is not None else np.nan
            ),
            "RecoveredSourcePath": str(p),
            "RecoveredSourcePriority": int(meta["Priority"]),
            "RecoveredSourceRow": int(idx),
        })

parsed = pd.DataFrame(parsed_rows)

print("Parsed clean CI rows:", len(parsed))

parsed.to_csv(
    OUT_DIR / "all_clean_recovered_ci_rows.csv",
    index=False,
)


Parsed clean CI rows: 229


## 6. Resolve one clean historical source per condition

Explicit `Direct-ShrinkAdaptive` comparisons and true `bootstrap.csv` files receive highest priority.
Only sign-consistent rows are eligible.


In [7]:
def comparison_score(s):
    s = str(s).lower().strip()

    if not s:
        return 2

    direct = "direct" in s
    finalish = any(
        token in s
        for token in [
            "shrinkadaptive",
            "shrink",
            "adaptive",
            "final",
            "ours",
        ]
    )

    if direct and finalish:
        return 0
    if direct:
        return 1
    return 2


def sign_of(v, tol=1e-12):
    if pd.isna(v):
        return 0
    if v > tol:
        return 1
    if v < -tol:
        return -1
    return 0


gain_lookup = base.set_index(KEY)["MSEGain_pct"].to_dict()

resolved_rows = []

if len(parsed):
    parsed["ComparisonScore"] = parsed["Comparison"].map(comparison_score)

    for key, g in parsed.groupby(KEY):
        frozen_gain = gain_lookup.get(key, np.nan)
        frozen_sign = sign_of(frozen_gain)

        g = g.copy()

        g["IntervalSign"] = np.select(
            [
                g["CI_Low_recovered"] > 0,
                g["CI_High_recovered"] < 0,
            ],
            [
                1,
                -1,
            ],
            default=0,
        )

        # Nonsignificant intervals can coexist with either observed direction.
        g["SignConsistent"] = (
            (g["IntervalSign"] == 0)
            | (frozen_sign == 0)
            | (g["IntervalSign"] == frozen_sign)
        )

        g = g[g["SignConsistent"]].copy()

        if not len(g):
            continue

        g["RepNumeric"] = pd.to_numeric(
            g["Replicates_recovered"],
            errors="coerce",
        ).fillna(-1)

        g = g.sort_values(
            [
                "ComparisonScore",
                "RecoveredSourcePriority",
                "RepNumeric",
                "RecoveredSourcePath",
            ],
            ascending=[True, True, False, True],
        )

        resolved_rows.append(g.iloc[0].to_dict())

resolved = pd.DataFrame(resolved_rows)

print("Clean resolved historical conditions:", len(resolved))

if len(resolved):
    display(
        resolved.sort_values(KEY)
    )

resolved.to_csv(
    OUT_DIR / "resolved_clean_historical_significance.csv",
    index=False,
)


Clean resolved historical conditions: 71


,Dataset,Backbone,Horizon,Comparison,MeanImprovement,CI_Low_recovered,CI_High_recovered,SigPositive_recovered,SigNegative_recovered,BlockLen_recovered,Replicates_recovered,RecoveredSourcePath,RecoveredSourcePriority,RecoveredSourceRow,ComparisonScore,IntervalSign,SignConsistent,RepNumeric
0,ETTh1,PatchTST,96,,NaN,-0.006288,-0.002074,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,8,2,-1,True,-1.0
1,ETTh1,PatchTST,192,,NaN,-0.009349,-0.004259,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,10,2,-1,True,-1.0
2,ETTh1,PatchTST,336,,NaN,-0.008331,-0.004493,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,12,2,-1,True,-1.0
3,ETTh1,PatchTST,720,,NaN,-0.022114,-0.014229,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,14,2,-1,True,-1.0
4,ETTh1,TimeMixer,96,,NaN,-0.003480,-0.000890,False,True,NaN,NaN,/data/dataset/strong_forecaster/three_backbone...,3,7,2,-1,True,-1.0
5,ETTh1,TimeMixer,192,,NaN,-0.006327,0.001269,False,False,NaN,NaN,/data/dataset/strong_forecaster/three_backbone...,3,17,2,0,True,-1.0
6,ETTh1,TimeMixer,336,,NaN,-0.011725,-0.004047,False,True,NaN,NaN,/data/dataset/strong_forecaster/three_backbone...,3,20,2,-1,True,-1.0
7,ETTh1,TimeMixer,720,,NaN,-0.039244,-0.025174,False,True,NaN,NaN,/data/dataset/strong_forecaster/three_backbone...,3,23,2,-1,True,-1.0
8,ETTh1,iTransformer,96,,-0.001978,-0.004286,0.000448,False,False,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,9,2,0,True,-1.0
9,ETTh1,iTransformer,192,,-0.006457,-0.009515,-0.002894,False,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3,11,2,-1,True,-1.0


## 7. Coalesce selected-table metadata with clean historical recovery

Selected Experiment 37 significance metadata are used first.
Recovered historical metadata fill only selected-table missing values.


In [8]:
if len(resolved):
    recovery_keep = [
        "Dataset",
        "Backbone",
        "Horizon",
        "CI_Low_recovered",
        "CI_High_recovered",
        "SigPositive_recovered",
        "SigNegative_recovered",
        "BlockLen_recovered",
        "Replicates_recovered",
        "RecoveredSourcePath",
        "RecoveredSourcePriority",
    ]

    rec = resolved[recovery_keep].copy()
else:
    rec = pd.DataFrame(
        columns=[
            "Dataset",
            "Backbone",
            "Horizon",
            "CI_Low_recovered",
            "CI_High_recovered",
            "SigPositive_recovered",
            "SigNegative_recovered",
            "BlockLen_recovered",
            "Replicates_recovered",
            "RecoveredSourcePath",
            "RecoveredSourcePriority",
        ]
    )

full = base.merge(
    rec,
    on=KEY,
    how="left",
    validate="one_to_one",
)

for c in ["CI_Low", "CI_High"]:
    full[c] = pd.to_numeric(full[c], errors="coerce")

full["Final_CI_Low"] = full["CI_Low"].combine_first(
    full["CI_Low_recovered"]
)

full["Final_CI_High"] = full["CI_High"].combine_first(
    full["CI_High_recovered"]
)

full["Final_SignificantPositive"] = (
    full["SignificantPositive"]
    .where(
        full["SignificantPositive"].notna(),
        full["SigPositive_recovered"],
    )
)

full["Final_SignificantNegative"] = (
    full["SignificantNegative"]
    .where(
        full["SignificantNegative"].notna(),
        full["SigNegative_recovered"],
    )
)

full["SignificanceSource"] = np.where(
    full["SelectedHasCI"],
    "Experiment37Selected",
    np.where(
        full["Final_CI_Low"].notna()
        & full["Final_CI_High"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["HasFinalCI"] = (
    full["Final_CI_Low"].notna()
    & full["Final_CI_High"].notna()
)

full["SignClass"] = np.select(
    [
        full["Final_CI_Low"] > 0,
        full["Final_CI_High"] < 0,
        full["HasFinalCI"],
    ],
    [
        "positive",
        "negative",
        "nonsignificant",
    ],
    default="missing",
)

full["ObservedDirection"] = np.select(
    [
        full["MSEGain_pct"] > 1e-12,
        full["MSEGain_pct"] < -1e-12,
    ],
    [
        "win",
        "loss",
    ],
    default="tie",
)

# Strict final sign audit.
bad = full[
    ((full["SignClass"] == "positive") & (full["MSEGain_pct"] <= 0))
    | ((full["SignClass"] == "negative") & (full["MSEGain_pct"] >= 0))
]

if len(bad):
    print("WARNING: sign-inconsistent final intervals:")
    display(bad[KEY + ["MSEGain_pct", "Final_CI_Low", "Final_CI_High", "SignificanceSource"]])
    raise RuntimeError("Final significance sign audit failed")

display(
    full.sort_values(KEY)
)

full.to_csv(
    OUT_DIR / "condition_level_96_with_clean_significance.csv",
    index=False,
)


,Dataset,Backbone,Horizon,ProtocolFamily,ExperimentFamily,Direct_MSE,Final_MSE,MSEGain_pct,Direct_MAE,Final_MAE,MAEGain_pct,FinalMeanAlpha,OracleHeadroom_pct,ValidationGain_pct,CI_Low,CI_High,SignificantPositive,SignificantNegative,SourcePath,SourceScore,RecoveredFamilyBonus,FinalSourceScore,SelectedHasCI,CI_Low_recovered,CI_High_recovered,SigPositive_recovered,SigNegative_recovered,BlockLen_recovered,Replicates_recovered,RecoveredSourcePath,RecoveredSourcePriority,Final_CI_Low,Final_CI_High,Final_SignificantPositive,Final_SignificantNegative,SignificanceSource,HasFinalCI,SignClass,ObservedDirection
0,ETTh1,PatchTST,96,other,cross_backbone_integrated_evidence,0.378632,NaN,-1.090730,0.400303,NaN,-1.181425,NaN,6.578487,NaN,-0.006288,-0.002074,False,True,/data/dataset/strong_forecaster/cross_backbone...,0,10000,10000,True,-0.006288,-0.002074,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3.0,-0.006288,-0.002074,False,True,Experiment37Selected,True,negative,loss
1,ETTh1,PatchTST,192,other,cross_backbone_integrated_evidence,0.413694,NaN,-1.676780,0.420978,NaN,-1.948489,NaN,6.327950,NaN,-0.009349,-0.004259,False,True,/data/dataset/strong_forecaster/cross_backbone...,0,10000,10000,True,-0.009349,-0.004259,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3.0,-0.009349,-0.004259,False,True,Experiment37Selected,True,negative,loss
2,ETTh1,PatchTST,336,other,cross_backbone_integrated_evidence,0.441984,NaN,-1.466894,0.441434,NaN,-1.700162,NaN,4.747864,NaN,-0.008331,-0.004493,False,True,/data/dataset/strong_forecaster/cross_backbone...,0,10000,10000,True,-0.008331,-0.004493,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3.0,-0.008331,-0.004493,False,True,Experiment37Selected,True,negative,loss
3,ETTh1,PatchTST,720,other,cross_backbone_integrated_evidence,0.461510,NaN,-3.931421,0.473867,NaN,-3.320312,NaN,4.868058,NaN,-0.022114,-0.014229,False,True,/data/dataset/strong_forecaster/cross_backbone...,0,10000,10000,True,-0.022114,-0.014229,NaN,True,NaN,NaN,/data/dataset/strong_forecaster/cross_backbone...,3.0,-0.022114,-0.014229,False,True,Experiment37Selected,True,negative,loss
4,ETTh1,SegMoE,96,segmoe_dataset_specific,etth1_full7_segmoe_oof_reranker,0.429677,0.434201,-1.052852,0.437933,0.441780,-0.878598,0.079709,9.609814,NaN,-0.006634,-0.002260,False,True,/data/dataset/strong_forecaster/etth1_full7_se...,1600,0,1600,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.006634,-0.002260,False,True,Experiment37Selected,True,negative,loss
5,ETTh1,SegMoE,192,segmoe_dataset_specific,etth1_full7_segmoe_oof_reranker,0.481695,0.492603,-2.264470,0.473062,0.480069,-1.481033,0.094744,7.912543,NaN,-0.013680,-0.008055,False,True,/data/dataset/strong_forecaster/etth1_full7_se...,1600,0,1600,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.013680,-0.008055,False,True,Experiment37Selected,True,negative,loss
6,ETTh1,SegMoE,336,segmoe_dataset_specific,etth1_full7_segmoe_oof_reranker,0.535394,0.547647,-2.288568,0.505612,0.512787,-1.418999,0.100000,7.730242,NaN,-0.014982,-0.009320,False,True,/data/dataset/strong_forecaster/etth1_full7_se...,1600,0,1600,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014982,-0.009320,False,True,Experiment37Selected,True,negative,loss
7,ETTh1,SegMoE,720,segmoe_dataset_specific,etth1_full7_segmoe_oof_reranker,0.657350,0.702025,-6.796243,0.571758,0.593895,-3.871679,0.200000,8.328258,NaN,-0.051040,-0.038125,False,True,/data/dataset/strong_forecaster/etth1_full7_se...,1600,0,1600,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.051040,-0.038125,False,True,Experiment37Selected,True,negative,loss
8,ETTh1,TimeMixer,96,other,three_backbone_integrated_evidence,0.388839,NaN,-0.567222,0.402163,NaN,-0.660399,NaN,6.073419,NaN,-0.003480,-0.000890,False,True,/data/dataset/strong_forecaster/three_backbone...,80,10000,10080,True,-0.003480,-0.000890,False,True,NaN,NaN,/data/dataset/strong_forecaster/three_backbone...,3.0,-0.003480,-0.000890,False,True,Experiment37Selected,True,negative,loss
9,ETTh1,TimeMixer,192,other,three_backbon

## 8. Final clean coverage summary

In [9]:
summary_ds = (
    full.groupby("Dataset")
    .agg(
        Conditions=("Horizon", "size"),
        WithCI=("HasFinalCI", "sum"),
        SignificantWins=("SignClass", lambda s: int((s == "positive").sum())),
        SignificantLosses=("SignClass", lambda s: int((s == "negative").sum())),
        NonSignificant=("SignClass", lambda s: int((s == "nonsignificant").sum())),
        Missing=("SignClass", lambda s: int((s == "missing").sum())),
    )
    .reset_index()
)

summary_bb = (
    full.groupby("Backbone")
    .agg(
        Conditions=("Horizon", "size"),
        WithCI=("HasFinalCI", "sum"),
        SignificantWins=("SignClass", lambda s: int((s == "positive").sum())),
        SignificantLosses=("SignClass", lambda s: int((s == "negative").sum())),
        NonSignificant=("SignClass", lambda s: int((s == "nonsignificant").sum())),
        Missing=("SignClass", lambda s: int((s == "missing").sum())),
    )
    .reset_index()
)

display(summary_ds)
display(summary_bb)

summary_ds.to_csv(
    OUT_DIR / "clean_significance_coverage_by_dataset.csv",
    index=False,
)

summary_bb.to_csv(
    OUT_DIR / "clean_significance_coverage_by_backbone.csv",
    index=False,
)

missing = full[~full["HasFinalCI"]].copy()

print("=" * 118)
print("EXPERIMENT 41 v3 — CLEAN SIGNIFICANCE AUDIT")
print("=" * 118)
print(f"Final CI coverage: {int(full['HasFinalCI'].sum())}/96")
print(f"Missing: {len(missing)}/96")
print(f"Significant wins: {int((full['SignClass'] == 'positive').sum())}")
print(f"Significant losses: {int((full['SignClass'] == 'negative').sum())}")
print(f"Non-significant: {int((full['SignClass'] == 'nonsignificant').sum())}")

print("\nSource breakdown:")
print(full["SignificanceSource"].value_counts(dropna=False).to_string())

print("\nDataset summary:")
print(summary_ds.to_string(index=False))

if len(missing):
    print("\nStill missing:")
    print(
        missing[
            KEY + ["MSEGain_pct", "ObservedDirection"]
        ].sort_values(KEY).to_string(index=False)
    )


,Dataset,Conditions,WithCI,SignificantWins,SignificantLosses,NonSignificant,Missing
0,ETTh1,16,16,0,14,2,0
1,Electricity,16,16,14,0,2,0
2,Exchange,16,16,1,0,15,0
3,Solar,16,16,14,0,2,0
4,Traffic,16,16,13,1,2,0
5,Weather,16,16,16,0,0,0


,Backbone,Conditions,WithCI,SignificantWins,SignificantLosses,NonSignificant,Missing
0,PatchTST,24,24,12,4,8,0
1,SegMoE,24,24,16,4,4,0
2,TimeMixer,24,24,15,3,6,0
3,iTransformer,24,24,15,4,5,0


EXPERIMENT 41 v3 — CLEAN SIGNIFICANCE AUDIT
Final CI coverage: 96/96
Missing: 0/96
Significant wins: 58
Significant losses: 15
Non-significant: 23

Source breakdown:
Experiment37Selected    68
HistoricalRecovery      28

Dataset summary:
    Dataset  Conditions  WithCI  SignificantWins  SignificantLosses  NonSignificant  Missing
      ETTh1          16      16                0                 14               2        0
Electricity          16      16               14                  0               2        0
   Exchange          16      16                1                  0              15        0
      Solar          16      16               14                  0               2        0
    Traffic          16      16               13                  1               2        0
    Weather          16      16               16                  0               0        0


## 9. Paper-ready significance table

This compact table is intended for the appendix. It reports only the 96 frozen downstream conditions
and the clean final confidence intervals. No post-hoc statistical label is synthesized from the gain
alone.


In [10]:
paper = full[
    [
        "Dataset",
        "Backbone",
        "Horizon",
        "MSEGain_pct",
        "Final_CI_Low",
        "Final_CI_High",
        "SignClass",
        "SignificanceSource",
    ]
].copy()

paper = paper.sort_values(
    ["Dataset", "Backbone", "Horizon"]
)

display(paper)

paper.to_csv(
    OUT_DIR / "paper_downstream_significance_96_conditions.csv",
    index=False,
)

latex = paper.copy()

for c in [
    "MSEGain_pct",
    "Final_CI_Low",
    "Final_CI_High",
]:
    latex[c] = latex[c].map(
        lambda v: "" if pd.isna(v) else f"{v:.6f}"
    )

(OUT_DIR / "paper_downstream_significance_96_conditions.tex").write_text(
    latex.to_latex(
        index=False,
        escape=False,
    ),
    encoding="utf-8",
)

print("Saved paper-ready CSV and LaTeX table.")


,Dataset,Backbone,Horizon,MSEGain_pct,Final_CI_Low,Final_CI_High,SignClass,SignificanceSource
0,ETTh1,PatchTST,96,-1.090730,-0.006288,-0.002074,negative,Experiment37Selected
1,ETTh1,PatchTST,192,-1.676780,-0.009349,-0.004259,negative,Experiment37Selected
2,ETTh1,PatchTST,336,-1.466894,-0.008331,-0.004493,negative,Experiment37Selected
3,ETTh1,PatchTST,720,-3.931421,-0.022114,-0.014229,negative,Experiment37Selected
4,ETTh1,SegMoE,96,-1.052852,-0.006634,-0.002260,negative,Experiment37Selected
5,ETTh1,SegMoE,192,-2.264470,-0.013680,-0.008055,negative,Experiment37Selected
6,ETTh1,SegMoE,336,-2.288568,-0.014982,-0.009320,negative,Experiment37Selected
7,ETTh1,SegMoE,720,-6.796243,-0.051040,-0.038125,negative,Experiment37Selected
8,ETTh1,TimeMixer,96,-0.567222,-0.003480,-0.000890,negative,Experiment37Selected
9,ETTh1,TimeMixer,192,-0.640128,-0.006327,0.001269,nonsignificant,Experiment37Selected


Saved paper-ready CSV and LaTeX table.


/tmp/ipykernel_111196/1286238842.py:37: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex.to_latex(
